In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_excel("../data/total_precipitation_daily_DISTRICT.xlsx")
df.head()
# df['total_precipitation'] = df['total_precipitation'] / 24.  # divide by 24h since it's hourly rainfall integrated over 1 day

,date,DISTRICT_S,total_precipitation
0,2018-01-01,Abo,0.019579
1,2018-01-01,Abong Mbang,0.016489
2,2018-01-01,Ako,NaN
3,2018-01-01,Akonolinga,0.009255
4,2018-01-01,Akwaya,0.000575


In [4]:
df = df[df['total_precipitation'] < 100.]  # filter out districts with nonsensical values
df.describe()

,total_precipitation
count,2.284390e+05
mean,1.438931e-01
std,3.120638e-01
min,1.000000e-08
25%,2.915755e-03
50%,3.476880e-02
75%,1.451384e-01
max,1.000818e+01


In [5]:
len(df[df['total_precipitation'] > 0.05])

101991

In [10]:
# we have integrated over a district, we need to divide by number of squares
from rasterstats import zonal_stats
import rasterio
import geopandas as gpd
import numpy as np

In [11]:
# Administrative level SHP file
admin_shp_path = "../data/Districts/District_sante_2022.shp"
gdf = gpd.read_file(admin_shp_path)
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.05, preserve_topology=True)

In [19]:
with rasterio.open("../data/precipitation.grib") as src:
    transform = src.transform
    gdf = gdf.to_crs(src.crs)
    data = src.read(1)
    data = np.ones(data.shape)
    
    # Perform zonal statistics for the admin districts
    stats = zonal_stats(
        gdf,  # District geometries
        data,  # Raster data (1st band of the GRIB file)
        affine=transform,  # Affine transform of the raster
        stats=["sum"],  # Statistics to calculate the sum of precipitation
        all_touched=True,  # Consider all pixels touched by the geometry
        nodata=0.0  # Set nodata value for raster
    )

    # Append the results for each district
    results = []
    for i, stat in enumerate(stats):
        district_name = gdf.iloc[i]['DISTRICT_S']
        results.append({
            'DISTRICT_S': district_name,
            'no_cells': stat['sum']  # Use sum as the aggregation type
        })
    results_df = pd.DataFrame(results)
    print(results_df)

      DISTRICT_S  no_cells
0            Abo      20.0
1    Abong Mbang      99.0
2            Ako      19.0
3     Akonolinga      55.0
4         Akwaya      33.0
..           ...       ...
195    Yokadouma     174.0
196         Yoko     229.0
197      Zoétele      16.0
198     Garoua 1       4.0
199   Kye - Ossi       6.0

[200 rows x 2 columns]


In [31]:
results_df.describe()

,no_cells
count,200.00000
mean,31.00500
std,36.57985
min,1.00000
25%,9.75000
50%,18.00000
75%,38.00000
max,229.00000


In [33]:
# divide by the number of raster cells in each district
df_norm = df.copy()
for district in df_norm['DISTRICT_S'].unique():
    no_cells = results_df.loc[results_df['DISTRICT_S'] == district, ['no_cells']].values[0]
    df_norm.loc[df_norm['DISTRICT_S'] == district, 'total_precipitation'] = df_norm[df_norm['DISTRICT_S'] == district]['total_precipitation'].values / no_cells

df_norm.describe()

,total_precipitation
count,2.284390e+05
mean,5.366333e-03
std,8.344115e-03
min,1.829268e-10
25%,2.311639e-04
50%,2.401158e-03
75%,8.053766e-03
max,2.283852e-01


In [42]:
len(df_norm[df_norm['total_precipitation'] > 0.05])

1043

In [70]:
df_norm['date'] = pd.to_datetime(df_norm['date'])
df_norm['total_precipitation'] = df_norm['total_precipitation'].astype(float)
for n in range(2, 22):
    # df_n = df_norm.groupby([pd.Grouper(key='date', freq=f'{n}D'), pd.Grouper('DISTRICT_S')])['total_precipitation'].mean()
    df_n = df_norm.groupby(['DISTRICT_S', 'date']).mean().reset_index(drop=True)#['total_precipitation'].rolling(n).mean()
    df_n = df_n.rolling(n).mean()
    # print(df_n)
    print(n, len(df_n[df_n['total_precipitation'] > 0.05]))

2 699
3 401
4 329
5 235
6 169
7 119
8 91
9 64
10 49
11 34
12 19
13 9
14 8
15 9
16 4
17 3
18 2
19 0
20 0
21 0


In [53]:
print(f"{df_norm['date'].min()} {df_norm['date'].max()}")

2018-01-01 00:00:00 2021-06-03 00:00:00
